In [2]:
# ==================== 1. 基础设置 ====================
import time
import numpy as np
import random
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

# 设置随机种子
def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(0)

# 设备设置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")


使用设备: cuda


In [3]:

# ==================== 2. 多头注意力 ====================
class MultiHeadAttentionFusion(nn.Module):
    """
    多头注意力融合机制 - 修复版本
    处理模态间的交互关系
    """
    def __init__(self, input_dims, num_heads=4, num_classes=5, dropout=0.1):
        super(MultiHeadAttentionFusion, self).__init__()
        self.num_modalities = len(input_dims)
        self.num_heads = num_heads
        
        # 确保统一维度能被num_heads整除
        self.unified_dim = 64
        assert self.unified_dim % num_heads == 0, "unified_dim必须能被num_heads整除"
        
        # 投影层：将不同模态映射到统一维度
        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim, self.unified_dim),
                nn.LayerNorm(self.unified_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ) for dim in input_dims
        ])
        
        # 查询、键、值的线性变换（用于多头注意力）
        self.q_proj = nn.Linear(self.unified_dim, self.unified_dim)
        self.k_proj = nn.Linear(self.unified_dim, self.unified_dim)
        self.v_proj = nn.Linear(self.unified_dim, self.unified_dim)
        
        # 输出投影
        self.out_proj = nn.Linear(self.unified_dim, self.unified_dim)
        
        # 缩放因子
        self.scale = (self.unified_dim // num_heads) ** -0.5
        
        # 前馈网络
        self.ffn = nn.Sequential(
            nn.Linear(self.unified_dim * self.num_modalities, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # 分类器
        self.classifier = nn.Linear(64, num_classes)
        
        # LayerNorm和Dropout
        self.norm1 = nn.LayerNorm(self.unified_dim)
        self.norm2 = nn.LayerNorm(self.unified_dim)
        self.dropout = nn.Dropout(dropout)
        
        # 模态重要性权重
        self.modal_importance = nn.Parameter(torch.ones(self.num_modalities) / self.num_modalities)
        
    def forward(self, *modalities):
        """
        前向传播
        Args:
            *modalities: 可变数量的模态特征张量
        Returns:
            output: 分类logits
            attention_weights: 注意力权重 [batch, num_heads, num_modalities, num_modalities]
            modal_importance: 模态重要性权重 [num_modalities]
        """
        batch_size = modalities[0].shape[0]
        
        # 1. 投影到统一维度
        projected_modals = []
        for i, modal in enumerate(modalities):
            projected = self.projections[i](modal)  # [batch, unified_dim]
            projected_modals.append(projected)
        
        # 2. 堆叠模态
        # 形状: [batch, num_modalities, unified_dim]
        modal_stack = torch.stack(projected_modals, dim=1)
        
        # 3. 准备多头注意力输入
        # 重塑为 [batch, num_modalities, num_heads, head_dim]
        head_dim = self.unified_dim // self.num_heads
        
        # 计算Q, K, V
        Q = self.q_proj(modal_stack).view(batch_size, self.num_modalities, self.num_heads, head_dim)
        K = self.k_proj(modal_stack).view(batch_size, self.num_modalities, self.num_heads, head_dim)
        V = self.v_proj(modal_stack).view(batch_size, self.num_modalities, self.num_heads, head_dim)
        
        # 转置以进行批量矩阵乘法
        # 形状: [batch, num_heads, num_modalities, head_dim]
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        
        # 4. 计算注意力权重
        # 计算QK^T / sqrt(d_k)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attention_weights = F.softmax(attention_scores, dim=-1)  # [batch, num_heads, num_modalities, num_modalities]
        
        # 5. 应用注意力权重
        attended = torch.matmul(attention_weights, V)  # [batch, num_heads, num_modalities, head_dim]
        
        # 6. 重塑回原始维度
        attended = attended.transpose(1, 2).contiguous().view(
            batch_size, self.num_modalities, self.unified_dim
        )
        
        # 7. 输出投影和残差连接
        attended = self.out_proj(attended)
        attended = self.dropout(attended)
        attended = self.norm1(modal_stack + attended)
        
        # 8. 应用模态重要性权重
        modal_weights = F.softmax(self.modal_importance, dim=0)  # [num_modalities]
        weighted_attended = attended * modal_weights.view(1, -1, 1)
        
        # 9. 展平并前馈网络
        flattened = weighted_attended.reshape(batch_size, -1)  # [batch, num_modalities * unified_dim]
        features = self.ffn(flattened)
        features = self.norm2(features)
        
        # 10. 分类
        output = self.classifier(features)
        
        return output, attention_weights, modal_weights

In [4]:
# ==================== 3. 简化的注意力融合类 ====================
class AttentionFusion(nn.Module):
    """改进的注意力融合，防止极端权重"""
    def __init__(self, input_dims, num_classes=5, temperature=1.0):
        super(AttentionFusion, self).__init__()
        total_dim = sum(input_dims)
        self.num_modalities = len(input_dims)
        self.temperature = temperature
        
        # 改进的注意力机制
        self.attention = nn.Sequential(
            nn.Linear(total_dim, 64),
            nn.LayerNorm(64),
            nn.GELU(),  # 使用GELU替代ReLU
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.GELU(),
            nn.Linear(32, self.num_modalities),
        )
        
        # 分类器
        self.classifier = nn.Sequential(
            nn.Linear(total_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
        # 初始化策略
        self._init_weights()
    
    def _init_weights(self):
        """更好的权重初始化"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)  # 小增益初始化
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, *modalities):
        # 拼接所有模态
        concatenated = torch.cat(modalities, dim=1)
        
        # 计算注意力分数
        attention_scores = self.attention(concatenated)
        
        # 应用温度参数和softmax
        attention_scores = attention_scores / self.temperature
        attn_weights = F.softmax(attention_scores, dim=1)
        
        # 应用注意力权重
        weighted_features = []
        start_idx = 0
        for i, modal in enumerate(modalities):
            modal_dim = modal.shape[1]
            end_idx = start_idx + modal_dim
            weighted = concatenated[:, start_idx:end_idx] * attn_weights[:, i:i+1]
            weighted_features.append(weighted)
            start_idx = end_idx
        
        # 再次拼接并分类
        final_features = torch.cat(weighted_features, dim=1)
        output = self.classifier(final_features)
        
        return output, attn_weights


In [5]:
# ==================== 4. 简化DNN模型 ====================
class SimpleDNN(nn.Module):
    """简化的DNN模型"""
    def __init__(self, input_dim, hidden_dim=128, output_dim=5):
        super(SimpleDNN, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        
        # 双分支结构
        self.branch_x = nn.Linear(input_dim, hidden_dim)
        self.branch_y = nn.Linear(input_dim, hidden_dim)
        
        # 分类头
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, output_dim)
        )
        
        self.norm = nn.LayerNorm(hidden_dim * 2)
        
    def forward(self, x):
        # 双分支处理
        x1 = self.branch_x(x)
        x2 = self.branch_y(x)
        
        # 拼接并归一化
        x_combined = torch.cat([x1, x2], dim=1)
        x_norm = self.norm(x_combined)
        
        # 分类
        output = self.classifier(x_norm)
        return output

# ==================== 5. EarlyStopping ====================
class EarlyStopping:
    """简化的早停机制"""
    def __init__(self, patience=10, delta=1e-4, path='checkpoint.pt'):
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False
        
    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                
    def save_checkpoint(self, model):
        torch.save(model.state_dict(), self.path)

# ==================== 6. BoostedDNN ====================
class BoostedDNN:
    """优化的Boosting DNN"""
    def __init__(self, model_class, n_estimators=5, lr=1e-4, 
                 hidden_dim=128, max_epochs=500, patience=20):
        self.model_class = model_class
        self.n_estimators = n_estimators
        self.lr = lr
        self.hidden_dim = hidden_dim
        self.max_epochs = max_epochs
        self.patience = patience
        self.models = []
        self.weights = []
        
    def fit(self, X, y, input_dim):
        """训练Boosting模型"""
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        X_tensor = torch.FloatTensor(X).to(device)
        y_tensor = torch.LongTensor(y).to(device)
        
        # 初始化残差（one-hot形式）
        n_classes = len(torch.unique(y_tensor))
        residual = F.one_hot(y_tensor, n_classes).float()
        
        for i in range(self.n_estimators):
            # 创建新模型
            model = self.model_class(input_dim, self.hidden_dim, n_classes).to(device)
            optimizer = optim.Adam(model.parameters(), lr=self.lr, weight_decay=1e-4)
            early_stop = EarlyStopping(patience=self.patience, 
                                      path=f'mid/boost_model_{i}.pt')
            
            # 训练当前模型拟合残差
            dataset = TensorDataset(X_tensor, residual)
            loader = DataLoader(dataset, batch_size=32, shuffle=True)
            
            model.train()
            for epoch in range(self.max_epochs):
                epoch_loss = 0
                for batch_X, batch_res in loader:
                    optimizer.zero_grad()
                    pred = model(batch_X)
                    loss = F.mse_loss(pred, batch_res)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    epoch_loss += loss.item()
                
                early_stop(epoch_loss/len(loader), model)
                if early_stop.early_stop:
                    break
            
            # 计算模型权重
            with torch.no_grad():
                model.eval()
                pred = model(X_tensor)
                weight = 1.0 / (torch.norm(pred - residual).item() + 1e-8)
                
            # 更新残差
            residual = residual - pred * 0.1  # 学习率衰减
            self.models.append(model)
            self.weights.append(weight)
            
            if (i+1) % 2 == 0:
                print(f"  Booster {i+1}/{self.n_estimators} trained, weight: {weight:.4f}")
    
    def predict(self, X):
        """集成预测"""
        if not self.models:
            raise ValueError("Model not trained yet!")
            
        device = next(self.models[0].parameters()).device
        X_tensor = torch.FloatTensor(X).to(device)
        
        # 加权投票
        total_pred = torch.zeros(len(X), self.models[0].output_dim).to(device)
        total_weight = sum(self.weights)
        
        for model, weight in zip(self.models, self.weights):
            with torch.no_grad():
                pred = model(X_tensor)
                total_pred += (weight / total_weight) * pred
        
        return total_pred.cpu().detach().numpy()


In [6]:
# ==================== 7. 数据加载与预处理 ====================
def load_and_preprocess_data():
    """加载并预处理数据"""
    # 加载数据（简化路径）
    data_paths = {
        'mrna': 'Datasets/mRNA.txt',
        'meth': 'Datasets/DNA methylation.txt', 
        'cnv': 'Datasets/CNV.txt',
        'label': 'Datasets/label.txt'
    }
    
    # 读取数据
    datasets = {}
    for name, path in data_paths.items():
        df = pd.read_csv(path, sep='\t')
        if 'gene_name' in df.columns:
            df.index = df['gene_name']
            df = df.drop('gene_name', axis=1)
        datasets[name] = df.T.values
    
    # 加载标签
    label_df = pd.read_csv(data_paths['label'], sep='\t')
    if 'id' in label_df.columns:
        label_df.index = label_df['id']
        label_df = label_df.drop('id', axis=1)
    
    labels, uniques = pd.factorize(label_df['subtype'])
    
    # 归一化
    scaler = preprocessing.MinMaxScaler()
    for name in ['mrna', 'meth', 'cnv']:
        datasets[name] = scaler.fit_transform(datasets[name])
    
    return datasets['mrna'], datasets['meth'], datasets['cnv'], labels, uniques



In [7]:
# ==================== 增强的中期融合：相关性增强的多头注意力 ====================

class SimpleCorrelationAttention(nn.Module):
    """
    简化版的相关性增强注意力 - 更稳定，易于训练
    """
    def __init__(self, input_dims, num_classes=5, dropout=0.1):
        super(SimpleCorrelationAttention, self).__init__()
        self.num_modalities = len(input_dims)
        
        # 模态特定编码器
        self.encoders = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim, 32),
                nn.LayerNorm(32),
                nn.ReLU(),
                nn.Dropout(dropout)
            ) for dim in input_dims
        ])
        
        # 相关性计算网络
        total_corr_pairs = self.num_modalities * (self.num_modalities - 1) // 2
        self.corr_net = nn.Sequential(
            nn.Linear(64, 32),  # 拼接两个模态的特征
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )
        
        # 注意力网络
        self.attention_net = nn.Sequential(
            nn.Linear(sum(input_dims), 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, self.num_modalities)
        )
        
        # 融合分类器
        self.classifier = nn.Sequential(
            nn.Linear(sum(input_dims), 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
        
    def compute_correlation_bias(self, encoded_modals):
        """计算基于相关性的注意力偏置"""
        batch_size = encoded_modals[0].shape[0]
        device = encoded_modals[0].device
        correlation_bias = torch.zeros(batch_size, self.num_modalities).to(device)
        
        # 计算每对模态的相关性
        idx = 0
        for i in range(self.num_modalities):
            for j in range(i+1, self.num_modalities):
                paired = torch.cat([encoded_modals[i], encoded_modals[j]], dim=1)
                corr = self.corr_net(paired).squeeze(-1)
                correlation_bias[:, i] += corr
                correlation_bias[:, j] += corr
                idx += 1
        
        # 归一化
        correlation_bias = correlation_bias / (self.num_modalities - 1)
        return correlation_bias
    
    def forward(self, *modalities):
        # 编码各模态
        encoded = []
        for i, modal in enumerate(modalities):
            encoded.append(self.encoders[i](modal))
        
        # 计算相关性偏置
        corr_bias = self.compute_correlation_bias(encoded)
        
        # 拼接原始特征
        concatenated = torch.cat(modalities, dim=1)
        
        # 计算基础注意力
        base_attention = self.attention_net(concatenated)
        
        # 结合相关性偏置
        enhanced_attention = base_attention + 0.1 * corr_bias
        attention_weights = F.softmax(enhanced_attention, dim=1)
        
        # 应用注意力
        weighted_features = []
        start_idx = 0
        for i, modal in enumerate(modalities):
            modal_dim = modal.shape[1]
            end_idx = start_idx + modal_dim
            weighted = concatenated[:, start_idx:end_idx] * attention_weights[:, i:i+1]
            weighted_features.append(weighted)
            start_idx = end_idx
        
        # 融合并分类
        fused = torch.cat(weighted_features, dim=1)
        output = self.classifier(fused)
        
        return output, attention_weights, corr_bias


# ==================== 对比实验函数 ====================

def compare_fusion_methods_with_correlation(modality_preds, y_test, modality_names):
    """
    对比不同融合方法的性能（使用简化版相关性注意力）
    """
    print("\n" + "="*70)
    print("对比不同融合方法的性能（含相关性增强注意力）")
    print("="*70)
    
    results = {}
    
    # 转换为tensor
    modality_tensors = [torch.FloatTensor(pred).to(device) for pred in modality_preds]
    y_test_tensor = torch.LongTensor(y_test).to(device)
    
    # 方法1: 简单平均融合
    print("\n1. 简单平均融合:")
    avg_pred = sum(modality_tensors) / len(modality_tensors)
    avg_acc = accuracy_score(y_test, torch.argmax(avg_pred, dim=1).cpu().numpy())
    results['平均融合'] = avg_acc
    print(f"   准确率: {avg_acc:.4f}")
    
    # 方法2: 加权融合
    print("\n2. 加权融合 (自适应权重):")
    modal_accs = []
    for i, pred in enumerate(modality_preds):
        acc = accuracy_score(y_test, np.argmax(pred, axis=1))
        modal_accs.append(acc)
    
    weights = np.array(modal_accs) / sum(modal_accs)
    print(f"   权重: {np.round(weights, 3)}")
    
    weighted_pred = sum(w * pred for w, pred in zip(weights, modality_tensors))
    weighted_acc = accuracy_score(y_test, torch.argmax(weighted_pred, dim=1).cpu().numpy())
    results['自适应加权融合'] = weighted_acc
    print(f"   准确率: {weighted_acc:.4f}")
    
    # 方法3: 基础注意力融合
    print("\n3. 基础注意力融合:")
    input_dims = [pred.shape[1] for pred in modality_preds]
    base_attn_fusion = AttentionFusion(input_dims).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(base_attn_fusion.parameters(), lr=0.001)
    
    base_attn_fusion.train()
    for epoch in range(300):
        optimizer.zero_grad()
        outputs, _ = base_attn_fusion(*modality_tensors)
        loss = criterion(outputs, y_test_tensor)
        loss.backward()
        optimizer.step()
    
    base_attn_fusion.eval()
    with torch.no_grad():
        outputs, attn_weights = base_attn_fusion(*modality_tensors)
        base_attn_acc = accuracy_score(y_test, torch.argmax(outputs, dim=1).cpu().numpy())
        results['基础注意力融合'] = base_attn_acc
        print(f"   准确率: {base_attn_acc:.4f}")
        print(f"   平均注意力权重: {attn_weights.mean(dim=0).cpu().numpy().round(3)}")
    
    # 方法4: 简化版相关性增强注意力
    print("\n4. 简化版相关性增强注意力:")
    simple_corr_fusion = SimpleCorrelationAttention(
        input_dims=input_dims,
        num_classes=5,
        dropout=0.2
    ).to(device)
    
    optimizer = optim.Adam(simple_corr_fusion.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    simple_corr_fusion.train()
    best_acc = 0
    for epoch in range(300):
        optimizer.zero_grad()
        outputs, attn_weights, corr_bias = simple_corr_fusion(*modality_tensors)
        loss = criterion(outputs, y_test_tensor)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(simple_corr_fusion.parameters(), 1.0)
        optimizer.step()
        scheduler.step(loss)
        
        if (epoch + 1) % 50 == 0:
            simple_corr_fusion.eval()
            with torch.no_grad():
                outputs, _, _ = simple_corr_fusion(*modality_tensors)
                acc = accuracy_score(y_test, torch.argmax(outputs, dim=1).cpu().numpy())
                if acc > best_acc:
                    best_acc = acc
                print(f"   Epoch {epoch+1}: Loss={loss.item():.4f}, Acc={acc:.4f}")
    
    results['相关性增强注意力'] = best_acc
    print(f"   最佳准确率: {best_acc:.4f}")
    
    # 显示最终注意力权重
    simple_corr_fusion.eval()
    with torch.no_grad():
        _, final_attn, final_corr = simple_corr_fusion(*modality_tensors)
        print(f"   平均注意力权重: {final_attn.mean(dim=0).cpu().numpy().round(3)}")
        print(f"   平均相关性偏置: {final_corr.mean(dim=0).cpu().numpy().round(3)}")
    
    return results

# ==================== 相关性分析工具函数 ====================

def analyze_modality_correlations(correlation_enhanced_model, modality_names, test_data):
    """
    分析模态间的相关性模式
    """
    print("\n" + "="*70)
    print("模态间相关性分析")
    print("="*70)
    
    model = correlation_enhanced_model
    correlation_matrix = model.last_correlation_matrix
    
    if correlation_matrix is not None:
        # 计算平均相关性矩阵
        avg_correlation = correlation_matrix.mean(axis=0)
        
        print("\n平均相关性矩阵:")
        print("    " + "  ".join([f"{name:6}" for name in modality_names]))
        for i, name in enumerate(modality_names):
            row = f"{name}: "
            for j in range(len(modality_names)):
                row += f"{avg_correlation[i, j]:.4f} "
            print(row)
        
        # 找出最强的跨模态相关性
        print("\n最强的跨模态相关性:")
        correlations = []
        for i in range(len(modality_names)):
            for j in range(i+1, len(modality_names)):
                corr = avg_correlation[i, j]
                correlations.append((modality_names[i], modality_names[j], corr))
        
        correlations.sort(key=lambda x: x[2], reverse=True)
        for modal1, modal2, corr in correlations:
            print(f"  {modal1} - {modal2}: {corr:.4f}")
        
        # 可视化建议
        print("\n融合策略建议:")
        if avg_correlation[0, 1] > 0.5:  # RNA和DNA高度相关
            print("  - RNA和DNA高度相关，可以共享部分特征提取器")
        if avg_correlation[0, 2] > 0.5:  # RNA和CNV高度相关
            print("  - RNA和CNV高度相关，可能存在共同的调控机制")
        if avg_correlation[1, 2] > 0.5:  # DNA和CNV高度相关
            print("  - DNA甲基化和CNV高度相关，可以考虑早期融合")
        
        # 注意力权重分析
        attention_weights = model.last_attention_weights
        if attention_weights is not None:
            print("\n注意力分布分析:")
            avg_attention = attention_weights.mean(axis=(0, 1))  # 平均所有batch和head
            print("  平均注意力矩阵:")
            for i, name_i in enumerate(modality_names):
                row = f"    {name_i}: "
                for j, name_j in enumerate(modality_names):
                    row += f"{avg_attention[i, j]:.4f} "
                print(row)


# ==================== 更新主函数 ====================

def main_with_correlation_enhanced():
    """
    使用相关性增强注意力的主训练流程
    """
    print("="*70)
    print("多模态分类系统 - 相关性增强注意力融合版本")
    print("="*70)
    
    # 1. 加载数据
    print("\n[1/4] 加载数据...")
    mrna_data, meth_data, cnv_data, labels, label_names = load_and_preprocess_data()
    
    # 2. 划分数据集
    print("[2/4] 划分数据集...")
    splits = {}
    modalities = ['RNA', 'DNA', 'CNV']
    data_dict = {'RNA': mrna_data, 'DNA': meth_data, 'CNV': cnv_data}
    
    for name, data in data_dict.items():
        X_train, X_test, y_train, y_test = train_test_split(
            data, labels, test_size=0.2, random_state=42, stratify=labels
        )
        splits[name] = {'train': X_train, 'test': X_test}
    
    _, _, y_train, y_test = train_test_split(
        labels, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    # 3. 训练单模态模型
    print("[3/4] 训练单模态Boosting模型...")
    modality_predictions = {}
    modality_accuracies = {}
    
    for modality in modalities:
        print(f"\n训练 {modality} 模型:")
        start_time = time.time()
        
        X_train = splits[modality]['train']
        X_test = splits[modality]['test']
        
        booster = BoostedDNN(
            model_class=SimpleDNN,
            n_estimators=5,
            lr=1e-4,
            hidden_dim=64,
            max_epochs=200,
            patience=10
        )
        
        booster.fit(X_train, y_train, X_train.shape[1])
        
        test_pred = booster.predict(X_test)
        train_pred = booster.predict(X_train)
        
        test_acc = accuracy_score(y_test, np.argmax(test_pred, axis=1))
        train_acc = accuracy_score(y_train, np.argmax(train_pred, axis=1))
        
        modality_accuracies[modality] = test_acc
        modality_predictions[modality] = test_pred
        
        print(f"  训练准确率: {train_acc:.4f}, 测试准确率: {test_acc:.4f}")
        print(f"  耗时: {time.time()-start_time:.1f}秒")
    
    # 4. 多模态融合对比实验
    print("\n[4/4] 多模态融合对比实验...")
    
    modality_preds = [modality_predictions[m] for m in modalities]
    
    # 运行增强的对比实验
    fusion_results = compare_fusion_methods_with_correlation(modality_preds, y_test, modalities)
    
    # 5. 打印总结
    print("\n" + "="*70)
    print("性能总结")
    print("="*70)
    
    print("\n单模态性能:")
    for modality in modalities:
        print(f"  {modality}: {modality_accuracies[modality]:.4f}")
    
    print("\n融合方法性能:")
    sorted_results = sorted(fusion_results.items(), key=lambda x: x[1], reverse=True)
    for method, acc in sorted_results:
        print(f"  {method}: {acc:.4f}")
    
    best_method, best_acc = sorted_results[0]
    print(f"\n最佳融合方法: {best_method} (准确率: {best_acc:.4f})")
    
    baseline_acc = fusion_results.get('平均融合', 0)
    improvement = (best_acc - baseline_acc) * 100
    print(f"相对于平均融合的提升: {improvement:.2f}%")
    
    print(f"\n标签类别: {label_names.tolist()}")
    print("="*70)


# 执行主程序
if __name__ == "__main__":
    main_with_correlation_enhanced()

多模态分类系统 - 相关性增强注意力融合版本

[1/4] 加载数据...
[2/4] 划分数据集...
[3/4] 训练单模态Boosting模型...

训练 RNA 模型:
  Booster 2/5 trained, weight: 0.6167
  Booster 4/5 trained, weight: 0.6654
  训练准确率: 1.0000, 测试准确率: 0.8019
  耗时: 15.1秒

训练 DNA 模型:
  Booster 2/5 trained, weight: 0.3076
  Booster 4/5 trained, weight: 0.1913
  训练准确率: 1.0000, 测试准确率: 0.6698
  耗时: 14.4秒

训练 CNV 模型:
  Booster 2/5 trained, weight: 0.1620
  Booster 4/5 trained, weight: 0.1926
  训练准确率: 0.9882, 测试准确率: 0.8491
  耗时: 15.2秒

[4/4] 多模态融合对比实验...

对比不同融合方法的性能（含相关性增强注意力）

1. 简单平均融合:
   准确率: 0.8491

2. 加权融合 (自适应权重):
   权重: [0.346 0.289 0.366]
   准确率: 0.8491

3. 基础注意力融合:
   准确率: 1.0000
   平均注意力权重: [0.356 0.271 0.374]

4. 简化版相关性增强注意力:
   Epoch 50: Loss=0.5130, Acc=0.8302
   Epoch 100: Loss=0.2186, Acc=0.9057
   Epoch 150: Loss=0.0302, Acc=1.0000
   Epoch 200: Loss=0.0038, Acc=1.0000
   Epoch 250: Loss=0.0013, Acc=1.0000
   Epoch 300: Loss=0.0007, Acc=1.0000
   最佳准确率: 1.0000
   平均注意力权重: [0.309 0.106 0.585]
   平均相关性偏置: [0.496 0.496 0.496]

性能总结

单模态性能: